# 00 — What Are We Building?

Imagine a tiny race engineer living in a laptop. You show it what the car has been doing, what the driver is pressing, what the track looks like, and what the weather feels like. It learns a small internal **make-believe racetrack**. Then you ask: “What happens if it rains harder?” or “What if the driver brakes later?” The engine imagines the next few seconds.

This notebook maps that child-friendly picture to the engineering words: **state, action, context, transition, world model, rollout, uncertainty, and scenario intervention**.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
import sys
sys.path.insert(0, str(ROOT / 'src'))

## The four pieces

- **State:** what the world is like now — speed, acceleration, track position, tyre age.
- **Action:** what the driver or controller does — throttle, brake, gear, DRS and an approximate steering signal.
- **Context:** conditions the driver cannot directly command — track curvature, temperature, rain, wind and grip.
- **Transition:** the rule that turns `(state now, action now, context now)` into `state next`.

A learned transition is the heart of our V1 world model.

In [ ]:
from apexsim.contracts import STATE_COLUMNS, ACTION_COLUMNS, CONTEXT_COLUMNS
print('STATE:', STATE_COLUMNS)
print('ACTION:', ACTION_COLUMNS)
print('CONTEXT:', CONTEXT_COLUMNS)

## What V1 can and cannot claim

V1 is a **telemetry dynamics simulator**, not a complete Formula 1 game engine. It predicts a compact state from recorded-style inputs. FastF1/OpenF1 do not expose every control and physical quantity available from F1 25 UDP, so V1 uses a steering proxy and cannot yet model suspension, tyre temperatures, damage, setup, energy deployment or wheel-level dynamics with game-level fidelity.

That limitation is not a weakness in the learning design: it creates a clean migration path. The canonical contract remains stable while richer F1 25 adapters fill more fields later.

In [ ]:
summary = json.loads((ROOT/'artifacts/runs/reference_gru/summary.json').read_text())
summary['model'], summary['metrics']['speed_mae_mps'] * 3.6

### Checkpoint
Explain the system without using the words neural network, tensor or latent. Then explain it again using those words. If both explanations describe the same causal flow, you understand the scope.